# Experiment 001 — baseline, on free-tier Colab

This notebook is a driver, not the implementation. Every algorithm lives in the
repository at `github.com/Cosmic-Witness/Sol`, which this notebook clones. Editing
code here would break reproducibility and would cost points under the code-quality
part of the rubric.

**What survives a disconnect**

| Artefact | Location | Why |
|---|---|---|
| Checkpoints | Google Drive | the runtime disk disappears with the runtime |
| Conditioning cache | Drive, as one zip | 707 small files sync slowly; one archive does not |
| Competition data | runtime disk | a re-download costs about two minutes |

Re-running every cell after a disconnect resumes training from the last completed
epoch. Nothing needs to be reset by hand.

In [16]:
# 1. Confirm a GPU is attached before spending time on setup.
#    Runtime > Change runtime type > T4 GPU, if this reports nothing.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 3 MiB
torch 2.11.0+cu128 | cuda available: True


In [17]:
# 2. Mount Drive. Everything that must outlive the session goes under DRIVE_ROOT.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/sol')
for sub in ('checkpoints', 'logs', 'outputs', 'archive'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print('drive root:', DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
drive root: /content/drive/MyDrive/sol


In [18]:
# 3. Clone the repository, or fast-forward it if the session already holds a copy.
import os
REPO_URL = 'https://github.com/Cosmic-Witness/Sol'
REPO_DIR = '/content/Sol'
BRANCH = 'exp-001-baseline'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only

os.chdir(REPO_DIR)
!git log --oneline -1

Already on 'exp-001-baseline'
Your branch is up to date with 'origin/exp-001-baseline'.
Already up to date.
cde9d57 (HEAD -> exp-001-baseline, origin/exp-001-baseline) Created using Colab


In [19]:
# 4. Install the packages Colab does not ship. Torch and OpenCV are already present,
#    so they are left alone; reinstalling torch costs minutes and gains nothing.
!pip install -q segmentation-models-pytorch albumentations pycocotools
print('installed')

installed


In [20]:
# 5. Fetch the competition data.
#
#    Place your kaggle.json at /content/drive/MyDrive/sol/kaggle.json beforehand.
#    The token is copied into place and never printed. Do not paste the key into
#    a cell: notebook output is saved with the file.
import os, shutil
from pathlib import Path

token_src = Path('/content/drive/MyDrive/sol/kaggle.json')
assert token_src.exists(), f'place your Kaggle API token at {token_src}'
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(token_src, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

DATA_DIR = Path('/content/data')
if not (DATA_DIR / 'MAGFiLO_1.0_Kaggle_2026').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c filament-segmentation-2026 -p {DATA_DIR}
    !cd {DATA_DIR} && unzip -q -o '*.zip' && rm -f *.zip

!ls {DATA_DIR}/MAGFiLO_1.0_Kaggle_2026
print('train images:', len(list((DATA_DIR / 'MAGFiLO_1.0_Kaggle_2026/train/train_images').glob('*.jpeg'))))
print('test images :', len(list((DATA_DIR / 'MAGFiLO_1.0_Kaggle_2026/test/test_images').glob('*.jpeg'))))

test  train
train images: 707
test images : 180


In [21]:
# 6. Write a Colab-specific config. The committed config.yaml stays untouched, so
#    the repository still describes the local run exactly.
import yaml
from pathlib import Path

cfg = yaml.safe_load(open('experiments/exp_001_baseline/config.yaml', encoding='utf-8'))
DATA_ROOT = '/content/data/MAGFiLO_1.0_Kaggle_2026'
cfg['paths'].update({
    'data_root': DATA_ROOT,
    'annotations': f'{DATA_ROOT}/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'cache_dir': '/content/cache',                       # local disk: fast random reads
    'checkpoint_dir': '/content/drive/MyDrive/sol/checkpoints',  # Drive: must survive
    'log_dir': '/content/drive/MyDrive/sol/logs',
    'output_dir': '/content/drive/MyDrive/sol/outputs',
})
# The T4 has 16 GB. Raise this only after watching nvidia-smi during epoch 0.
cfg['data']['num_workers'] = 2

COLAB_CONFIG = 'experiments/exp_001_baseline/config_colab.yaml'
yaml.safe_dump(cfg, open(COLAB_CONFIG, 'w', encoding='utf-8'), sort_keys=False)
print(open(COLAB_CONFIG, encoding='utf-8').read())

experiment: exp_001_baseline
seed: 2026
paths:
  data_root: /content/data/MAGFiLO_1.0_Kaggle_2026
  annotations: /content/data/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json
  cache_dir: /content/cache
  checkpoint_dir: /content/drive/MyDrive/sol/checkpoints
  log_dir: /content/drive/MyDrive/sol/logs
  output_dir: /content/drive/MyDrive/sol/outputs
data:
  image_size: 512
  val_fraction: 0.15
  num_workers: 2
preprocess:
  clahe_clip: 2.0
  clahe_grid: 8
  blur_sigma: 0.7
model:
  arch: unetplusplus
  encoder: efficientnet-b4
  encoder_weights: imagenet
  in_channels: 3
  classes: 1
loss:
  bce_weight: 0.5
  dice_weight: 0.5
  pos_weight: 8.0
train:
  epochs: 60
  batch_size: 8
  grad_accum_steps: 1
  lr: 0.0001
  weight_decay: 0.0001
  scheduler: cosine
  warmup_epochs: 2
  amp: true
  checkpoint_every_epoch: true
  early_stopping_patience: 12
  pq_subset: 60
postprocess:
  threshold: 0.5
  min_area: 150
  closing_kernel: 1
  dilate_iterations: 0



In [22]:
# 7. Conditioning cache: restore from Drive if it exists, otherwise build and archive.
#
#    Building costs about 20 minutes for 707 observations at roughly 1.6 s each.
#    The archive means that price is paid once, not once per session.
from pathlib import Path

CACHE_DIR = Path('/content/cache')
ARCHIVE = Path('/content/drive/MyDrive/sol/archive/cache_512.zip')

if ARCHIVE.exists() and not CACHE_DIR.exists():
    print('restoring cache from Drive')
    !mkdir -p {CACHE_DIR} && unzip -q {ARCHIVE} -d {CACHE_DIR}

!python -m experiments.exp_001_baseline.src.prepare_cache --config {COLAB_CONFIG}

if not ARCHIVE.exists():
    print('archiving cache to Drive')
    !cd {CACHE_DIR} && zip -q -r {ARCHIVE} images masks
    print('archived:', ARCHIVE.stat().st_size / 1e6, 'MB')

loading annotations into memory...
Done (t=0.74s)
creating index...
index created!
cache 100/1154
cache 200/1154
cache 300/1154
cache 400/1154
cache 500/1154
cache 600/1154
cache 700/1154
cache 800/1154
cache 900/1154
cache 1000/1154
cache 1100/1154
cache ready: 707 images, 1154 masks


In [23]:
# 8. Train. Re-run this cell after any disconnect; it resumes from last.pt.
#
#    Keep this browser tab open and stop the machine from sleeping. Free-tier
#    Colab has no background execution, so a closed tab ends the run.
!python -m experiments.exp_001_baseline.src.train --config {COLAB_CONFIG}

device: cuda
train: 601 observations / 974 records | val: 106 observations / 180 records
/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
loading annotations into memory...
Done (t=0.63s)
creating index...
index created!
^C


In [24]:
# 9. Score the best checkpoint over the whole validation fold, at native 2048.
#    This is the number that goes into RESULTS.md, not the per-epoch subset value.
!python -m experiments.exp_001_baseline.src.evaluate \
    --config {COLAB_CONFIG} \
    --checkpoint /content/drive/MyDrive/sol/checkpoints/best.pt

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Sol/experiments/exp_001_baseline/src/evaluate.py", line 170, in <module>
    main()
    ~~~~^^
  File "/content/Sol/experiments/exp_001_baseline/src/evaluate.py", line 134, in main
    state = torch.load(args.checkpoint, map_location=device, weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ~~~~^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive

In [25]:
# 10. Predict the test set and write the submission. The overlap validator runs
#     at the end of this script and raises if any two masks share a pixel.
!python -m experiments.exp_001_baseline.src.predict \
    --config {COLAB_CONFIG} \
    --checkpoint /content/drive/MyDrive/sol/checkpoints/best.pt \
    --images /content/data/MAGFiLO_1.0_Kaggle_2026/test/test_images \
    --output /content/drive/MyDrive/sol/outputs/submission.csv \
    --tta

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Sol/experiments/exp_001_baseline/src/predict.py", line 151, in <module>
    main()
    ~~~~^^
  File "/content/Sol/experiments/exp_001_baseline/src/predict.py", line 93, in main
    state = torch.load(args.checkpoint, map_location=device, weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ~~~~^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/My

## After this notebook

`submission.csv` and `best.pt` now sit in `MyDrive/sol/`. Both get pulled to the
local machine from there. The submission is made from the laptop, not from this
notebook, so the overlap validator and the local PQ check run one more time
against the exact file that is uploaded.